Question 1.

Step 1. Removing not necessary info in the table and changing columns names

In [2]:
import pandas as pd
import numpy as np

def load_energy_data():
    energy = pd.read_excel('C:/Users/orban/Desktop/Energy Indicators.xls',
                           skiprows=17,
                           skipfooter=38)

    energy = energy.iloc[:, 2:]

    energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable']

    return energy

energy = load_energy_data()
energy.head()

,Country,Energy Supply,Energy Supply per Capita,% Renewable
0,Afghanistan,321,10,78.669280
1,Albania,102,35,100.000000
2,Algeria,1959,51,0.551010
3,American Samoa,...,...,0.641026
4,Andorra,9,121,88.695650


Step 2. Removing '...' to NaN values

In [3]:
energy.replace('...', np.nan, inplace=True)

,Country,Energy Supply,Energy Supply per Capita,% Renewable
0,Afghanistan,321,10,78.669280
1,Albania,102,35,100.000000
2,Algeria,1959,51,0.551010
3,American Samoa,NaN,NaN,0.641026
4,Andorra,9,121,88.695650
...,...,...,...,...
222,Viet Nam,2554,28,45.321520
223,Wallis and Futuna Islands,0,26,0.000000
224,Yemen,344,13,0.000000
225,Zambia,400,26,99.714670


Step 3. Changing energy column type to numeric

In [4]:
energy['Energy Supply'] = pd.to_numeric(energy['Energy Supply'])
energy['Energy Supply per Capita'] = pd.to_numeric(energy['Energy Supply per Capita'])

Step 4. Changing petajoules to gigajoules

In [5]:
energy['Energy Supply'] = energy['Energy Supply'] * 1000000

Step 5. Renaming countries

In [6]:
energy['Country'] = energy['Country'].str.rstrip('0123456789')

energy['Country'] = energy['Country'].str.split('(').str[0].str.strip()

country_mapping = {
    "Republic of Korea": "South Korea",
    "United States of America": "United States",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "China, Hong Kong Special Administrative Region": "Hong Kong"
}
energy['Country'] = energy['Country'].replace(country_mapping)

Step 6. Loading GDP data and changing countries names

In [17]:
GDP = pd.read_csv("C:/Users/orban/Desktop/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_133326/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_133326.csv", skiprows=4)

gdp_country_mapping = {
    "Korea, Rep.": "South Korea",
    "Iran, Islamic Rep.": "Iran",
    "Hong Kong SAR, China": "Hong Kong"
}

GDP['Country Name'] = GDP['Country Name'].replace(gdp_country_mapping)

GDP = GDP.rename(columns={'Country Name': 'Country'})

columns_to_keep = ['Country', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']
GDP_filtered = GDP[columns_to_keep]

GDP_filtered.head()

,Country,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
0,Aruba,2.469783e+09,2.677641e+09,2.843025e+09,2.553793e+09,2.453597e+09,2.637859e+09,2.615208e+09,2.727849e+09,2.790850e+09,2.962907e+09
1,Africa Eastern and Southern,5.875005e+11,6.749999e+11,7.267704e+11,7.269216e+11,8.638221e+11,9.598225e+11,9.692657e+11,9.813089e+11,9.988362e+11,9.100020e+11
2,Afghanistan,6.971758e+09,9.747886e+09,1.010930e+10,1.241615e+10,1.585667e+10,1.780510e+10,1.990733e+10,2.014642e+10,2.049713e+10,1.913422e+10
3,Africa Western and Central,4.020774e+11,4.709476e+11,5.742598e+11,5.150371e+11,6.057943e+11,6.911334e+11,7.480300e+11,8.441804e+11,9.040719e+11,7.780221e+11
4,Angola,5.865366e+10,7.303782e+10,9.879043e+10,8.170518e+10,9.554692e+10,1.255516e+11,1.435729e+11,1.488452e+11,1.534499e+11,1.025431e+11


Step 7. Loading the "Sciamgo Journal and Country Rank data for Energy Engineering and Power Technology"

In [13]:
ScimEn = pd.read_excel("C:/Users/orban/Desktop/scimagojr country rank 1996-2024.xlsx")
ScimEn_top15 = ScimEn[ScimEn['Rank'] <= 15]

ScimEn.head()

,Rank,Country,Region,Documents,Citable documents,Citations,Self-citations,Citations per document,H index
0,1,China,Asiatic Region,472465,470142,6591474,4594673,13.95,380
1,2,United States,Northern America,222772,217929,4131736,1119917,18.55,491
2,3,India,Asiatic Region,96975,94714,1243636,479638,12.82,258
3,4,United Kingdom,Western Europe,61946,60287,1326766,207337,21.42,313
4,5,Japan,Asiatic Region,61939,61307,813472,166907,13.13,246


Step 8. Merging tables

In [19]:
merged_df = pd.merge(ScimEn_top15, energy, how='inner', on='Country')

final_df = pd.merge(merged_df, GDP_filtered, how='inner', on='Country')

final_df = final_df.set_index('Country')

Top15 = final_df
print(Top15.shape)
Top15.head()

(15, 21)


,Rank,Region,Documents,Citable documents,Citations,Self-citations,Citations per document,H index,Energy Supply,Energy Supply per Capita,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
Country,,,,,,,,,,,,,,,,,,,,,
China,1,Asiatic Region,472465,470142,6591474,4594673,13.95,380,1.271910e+11,93.0,...,2.791498e+12,3.604056e+12,4.667346e+12,5.189577e+12,6.192565e+12,7.671757e+12,8.673665e+12,9.743124e+12,1.067453e+13,1.128081e+13
United States,2,Northern America,222772,217929,4131736,1119917,18.55,491,9.083800e+10,286.0,...,1.381559e+13,1.447423e+13,1.476986e+13,1.447806e+13,1.504896e+13,1.559973e+13,1.625397e+13,1.684319e+13,1.755068e+13,1.820602e+13
India,3,Asiatic Region,96975,94714,1243636,479638,12.82,258,3.319500e+10,26.0,...,9.402599e+11,1.216736e+12,1.198895e+12,1.341888e+12,1.675616e+12,1.823052e+12,1.827638e+12,1.856722e+12,2.039126e+12,2.103588e+12
United Kingdom,4,Western Europe,61946,60287,1326766,207337,21.42,313,7.920000e+09,124.0,...,2.719558e+12,3.104700e+12,2.945252e+12,2.429358e+12,2.496741e+12,2.675590e+12,2.719716e+12,2.796908e+12,3.085362e+12,2.945580e+12
Japan,5,Asiatic Region,61939,61307,813472,166907,13.13,246,1.898400e+10,149.0,...,4.601663e+12,4.579751e+12,5.106679e+12,5.289493e+12,5.759072e+12,6.233147e+12,6.272363e+12,5.212328e+12,4.896994e+12,4.444931e+12


Question 2.

In [25]:
def answer_two():
    Top15 = final_df
    years = ['2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']

    # Convert all year columns to numeric values to avoid TypeError
    Top15[years] = Top15[years].apply(pd.to_numeric, errors='coerce')

    # Calculate the mean across the rows (axis=1) and sort descending
    avgGDP = Top15[years].mean(axis=1).sort_values(ascending=False)

    # Rename the Series
    avgGDP = avgGDP.rename('avgGDP')

    return avgGDP

# Call the function and print the result
gdp_list = answer_two()
print(gdp_list)

Country
United States         1.570403e+13
China                 7.048894e+12
Japan                 5.239642e+12
Germany               3.590126e+12
United Kingdom        2.791877e+12
France                2.692000e+12
Italy                 2.152983e+12
Brazil                1.988889e+12
Russian Federation    1.666746e+12
Canada                1.616359e+12
India                 1.602352e+12
Spain                 1.406644e+12
South Korea           1.275614e+12
Australia             1.211998e+12
Iran                  4.567516e+11
Name: avgGDP, dtype: float64


Question 3.

In [26]:
def answer_three():
    Top15 = final_df
    avgGDP = answer_two()

    sixth_country = avgGDP.index[5]

    gdp_change = Top15.loc[sixth_country, '2015'] - Top15.loc[sixth_country, '2006']

    return gdp_change

print(answer_three())

124621907951.68018


Question 4.

In [27]:
def answer_four():

    Top15['Citation Ratio'] = Top15['Self-citations'] / Top15['Citations']

    max_ratio = Top15['Citation Ratio'].max()

    country_max = Top15['Citation Ratio'].idxmax()

    return (country_max, max_ratio)

print(answer_four())

('China', np.float64(0.6970630544852335))


Question 5.

In [28]:
def answer_five():

    Top15['Population Estimate'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']

    sorted_population = Top15['Population Estimate'].sort_values(ascending=False)

    third_country = sorted_population.index[2]

    return third_country

print(answer_five())

United States


Question 6.

In [29]:
def answer_six():

    Top15['Population Estimate'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']

    Top15['Citable docs per Capita'] = Top15['Citable documents'] / Top15['Population Estimate']

    Top15['Citable docs per Capita'] = Top15['Citable docs per Capita'].astype(float)
    Top15['Energy Supply per Capita'] = Top15['Energy Supply per Capita'].astype(float)

    correlation = Top15['Citable docs per Capita'].corr(Top15['Energy Supply per Capita'])

    return correlation

print(answer_six())

0.6905473831164103


Question 7.

In [30]:
def answer_seven():

    Top15['Population Estimate'] = (Top15['Energy Supply'] / Top15['Energy Supply per Capita']).astype(float)

    ContinentDict  = {'China':'Asia',
                      'United States':'North America',
                      'Japan':'Asia',
                      'United Kingdom':'Europe',
                      'Russian Federation':'Europe',
                      'Canada':'North America',
                      'Germany':'Europe',
                      'India':'Asia',
                      'France':'Europe',
                      'South Korea':'Asia',
                      'Italy':'Europe',
                      'Spain':'Europe',
                      'Iran':'Asia',
                      'Australia':'Australia',
                      'Brazil':'South America'}

    continent_stats = Top15['Population Estimate'].groupby(ContinentDict).agg(['size', 'sum', 'mean', 'std'])

    continent_stats.index.name = 'Continent'

    return continent_stats

print(answer_seven())

               size           sum          mean           std
Continent                                                    
Asia              5  2.898666e+09  5.797333e+08  6.790979e+08
Australia         1  2.331602e+07  2.331602e+07           NaN
Europe            6  4.579297e+08  7.632161e+07  3.464767e+07
North America     2  3.528552e+08  1.764276e+08  1.996696e+08
South America     1  2.059153e+08  2.059153e+08           NaN
